# Vectors, Matrices & Operations

**Module**: 00 Mathematical Foundations · **Topic**: Linear Algebra · **Notebook**: 01  
**Prerequisites**: None · **Estimated time**: 60 min  
**Priority band**: P1 Diagnostics — *Does this person have the condition? How sure are we?*  
**Goal**: `predict` — Predict — the model's job is accuracy on unseen patients; coefficients are not mechanisms.  
**GDS division**: 3 Computing with data  
**Dataset**: `screening`

## Learning Objectives

By the end of this notebook you will be able to:

1. Write a patient as a vector and a lab panel as a matrix, and say what a row, a column and the shape mean clinically.
2. Compute dot products, lengths, Euclidean distances and cosine similarities, first as loops and then as matrix products, and say what each one measures about two patients.
3. Explain why the units a panel is recorded in decide who counts as "similar", and standardise using the seen (training) patients only.
4. Detect a redundant test from a panel's rank, singular values and condition number, and say what redundancy does to coefficients and what it leaves alone.
5. Turn similarity into a prediction, either a nearest-neighbour vote or a matrix–vector score, and judge it on new patients with sensitivity and specificity.

In [ ]:
import sys
from pathlib import Path
# Jupyter starts the kernel in this notebook's folder; put the repo root on the path so shared/ and datasets/ import
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from shared.plotting import set_style, PALETTE, grid
from shared.explanation import rfi, clinical_reading, three_registers
from datasets import load_screening
from datasets.baselines import clinician_baseline
from sklearn.metrics import roc_auc_score, roc_curve

set_style()
SEED = 42
np.random.seed(SEED)
HEALTHY, DISEASED = PALETTE[0], PALETTE[1]   # one colour per disease status, used in every figure

---
## 1. The Clinical Problem

🗣️ **Plain** — A GP gets a new patient's twelve-test screening panel back. The question is the band's question: *does this person have the condition, and how sure are we?* One way to answer it is the way a clinician draws on the patients they remember: "who have I seen before who looked like this, and how did they turn out?" A computer needs two things to do the same. It needs a way to write a patient down as numbers, and a way to measure how alike two patients are. This notebook builds both, then breaks them on purpose, three different ways.

**The data.** There are 2,000 synthetic patients from `datasets.load_screening`, with 12 tests each, recorded in SD units (0 = average, 1 = one standard deviation above average), and 15% have the condition. The generator built in:
- two strong tests (`bm_01`, `bm_02`) and three weak ones (`bm_03`–`bm_05`);
- six tests that carry no signal (`bm_06`–`bm_11`);
- one near-copy, `bm_12` ≈ `bm_01`.

We know this only because the data are synthetic (`return_truth=True`), and every method below is checked against it. First, look at the panel.

In [ ]:
# VISUAL 1 — the raw clinical data
df, truth = load_screening(return_truth=True)           # truth = what the generator built in, for checking later
BM = [c for c in df.columns if c.startswith("bm_")]
X_all = df[BM].to_numpy()                               # the panel as a matrix: one row per patient, one column per test
y_all = df["disease"].to_numpy()
print(f"X is {X_all.shape[0]} patients × {X_all.shape[1]} tests; prevalence {y_all.mean():.1%}")
print(f"patient 0 as a vector: {np.round(X_all[0], 2)}")
print(f"generator truth: {truth}")

rng = np.random.default_rng(SEED)                       # 30 patients from each group so the eye can compare rows
show = np.r_[rng.choice(np.where(y_all == 1)[0], 30, replace=False), rng.choice(np.where(y_all == 0)[0], 30, replace=False)]

fig, ax = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={"width_ratios": [1.25, 1]})
sns.heatmap(X_all[show], cmap="RdBu_r", center=0, vmin=-3, vmax=3, ax=ax[0], xticklabels=BM, yticklabels=False,
            cbar_kws={"label": "result (SD units)"})
ax[0].axhline(30, color="k", lw=2)
ax[0].set_title("Diseased rows run warm in bm_01, bm_02 and bm_12;\nmost other columns look the same in both groups")
ax[0].set(xlabel="test (one column each)", ylabel="patients: 30 diseased (top), 30 healthy")
for d, lab, col in [(0, "healthy", HEALTHY), (1, "diseased", DISEASED)]:
    m = y_all == d
    ax[1].scatter(X_all[m, 0], X_all[m, 1], s=10, alpha=0.45, color=col, label=f"{lab} (n={m.sum()})")
ax[1].set_title("Two tests already pull most diseased patients up and right,\nbut the groups still overlap")
ax[1].set(xlabel="bm_01 (SD units)", ylabel="bm_02 (SD units)")
ax[1].legend(loc="upper left")
plt.tight_layout(); plt.show()

---
## 2. What Is Vectors, Matrices & Operations?

🗣️ **Plain**

Write a patient's twelve test results down in a fixed order and you have a **vector**: an ordered list of numbers, one per test, always in the same order. Stack the lists for 2,000 patients, one patient per line, and you have a **matrix**: a table with patients as rows and tests as columns, the same shape as a results spreadsheet. Now picture each patient as an arrow. It starts at the *average* patient and points towards that person's own pattern of results: long if the results are far from average, short if they are close. To compare two patients, line their results up test by test, multiply each matching pair and add the answers. That total is the **dot product**. It is large when both patients are high (or both low) on the same tests, near zero when their results have nothing in common, and negative when one is high where the other is low. **Cosine similarity** is the dot product with the arrows' lengths divided out. It asks only "do these two arrows point the same way?", so the same pattern counts as a match whether it is mild or severe. **Distance** asks how far apart the two arrow tips are. A new patient whose arrow points the way the diseased patients' arrows point is probably diseased. The catch: the arrows depend on the units each test is written in, so the units quietly decide who counts as similar.

💻 **Developer**

```
X  : float array, shape (n_patients, n_tests); row i is patient i          memory 8·n·p bytes
x @ y                     dot product                                        O(p)
sqrt(x @ x)               length of a patient vector                         O(p)
(x @ y) / (|x| |y|)       cosine similarity
Xn = X / |each row|       every row to length 1
C  = Xn_new @ Xn_seen.T   all cosines at once                                O(n_new · n_seen · p) time, O(n_new · n_seen) memory
D2 = |a|² + |b|² − 2ABᵀ   all squared distances, no Python loop              same cost
s  = X @ w                one linear risk score per patient                  O(n · p)
kNN risk = mean(y_seen[top-k of each row of C])                              O(n_new · n_seen · log n_seen) to sort
rank, condition number via the SVD of X                                      O(n · p²)
```

NumPy: `@`, `np.linalg.norm`, `np.linalg.svd`, `np.linalg.matrix_rank`, `np.linalg.cond`, `np.linalg.lstsq`. scikit-learn: `StandardScaler`, `cosine_similarity`, `euclidean_distances`, `KNeighborsClassifier(metric="cosine", algorithm="brute")`. The all-pairs matrix here is 600 × 1,400 float64, which is 6.7 MB. At 100,000 × 100,000 it would be 80 GB, so compute it in blocks of rows or use an approximate nearest-neighbour index.

🔬 **Technical**

A patient is $x \in \mathbb{R}^p$; a panel is $X \in \mathbb{R}^{n \times p}$ with rows $x_i^\top$. The Euclidean inner product is $\langle x, y\rangle = x^\top y = \sum_{j=1}^{p} x_j y_j$, with norm $\|x\|_2 = \langle x, x\rangle^{1/2}$. The Cauchy–Schwarz inequality $|\langle x, y\rangle| \le \|x\|\,\|y\|$ makes $\cos\theta = \langle x, y\rangle / (\|x\|\,\|y\|) \in [-1, 1]$ well defined for non-zero vectors. Distances follow from inner products: $\|x - y\|^2 = \|x\|^2 + \|y\|^2 - 2\langle x, y\rangle$. Every pairwise quantity in this notebook is therefore a function of the Gram matrix $G = XX^\top$, which is symmetric positive semidefinite. On unit vectors $\|x - y\|^2 = 2(1 - \cos\theta)$, so the cosine and Euclidean rankings coincide once rows are normalised. Rescaling the tests by $D = \mathrm{diag}(d_1, \dots, d_p)$ replaces $\langle x, y\rangle$ with $x^\top D^2 y$. That is a different inner product, and so a different set of nearest neighbours. Euclidean distance is invariant under orthogonal maps ($Q^\top Q = I$), not under diagonal rescaling. For mean-centred columns $u, v$, $\cos(u, v)$ is their Pearson correlation. $\mathrm{rank}(X)$ is the dimension of the column space, equal to the number of non-zero singular values. If one column is a linear combination of the others, then $\mathrm{rank}(X) < p$, $X^\top X$ is singular, and the least-squares solutions form an affine set $w^\ast + \ker X$, of which `lstsq` returns the minimum-norm member. The condition number $\kappa(X) = \sigma_{\max} / \sigma_{\min}$ measures how much a small relative change in the data can be amplified in the solution. Forming $X^\top X$ squares it: $\kappa(X^\top X) = \kappa(X)^2$. **Assumptions:**
- the chosen inner product encodes which tests matter and by how much;
- cosine assumes the origin (here, the seen-patient mean) is meaningful, and discards magnitude;
- a nearest-neighbour vote assumes that patients with nearby panels share disease risk.

(Deisenroth, Faisal & Ong, 2020, chapters 2–4.)

---
## 3. Visual Intuition

Two tests and three illustrative patients carry every idea in this notebook. Patients are arrows. Similarity of pattern is the angle between two arrows. The dot product is one arrow's length times the other's shadow on it. Severity is length, which cosine ignores and distance does not.

In [ ]:
# VISUAL 2 — concept diagram: two tests, three illustrative patients (SD units; chosen by hand to make the geometry visible)
A, B = np.array([2.0, 0.8]), np.array([1.0, 1.9])
C = 0.4 * A                                              # the same pattern as A, milder
cos_AB = A @ B / (np.linalg.norm(A) * np.linalg.norm(B))
shadow = (A @ B) / (A @ A) * A                           # B projected onto A's direction

def arrow(ax, v, col, lab, dy=0.06):
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", lw=2.5, color=col))
    ax.text(v[0] + 0.06, v[1] + dy, lab, color=col, fontsize=13, fontweight="bold")

fig, axes = grid(3, ncols=3, panel=(5.8, 5.4))
arrow(axes[0], A, DISEASED, "A"); arrow(axes[0], B, PALETTE[2], "B")
axes[0].plot(*zip(A, B), "--", color=PALETTE[7], lw=2, label=f"gap between the tips = {np.linalg.norm(A - B):.2f}")
axes[0].set_title(f"Angle and gap are two different questions:\ncos θ = {cos_AB:.2f} (similar pattern), gap = {np.linalg.norm(A - B):.2f}")
arrow(axes[1], A, DISEASED, "A"); arrow(axes[1], B, PALETTE[2], "B")
axes[1].plot(*zip(B, shadow), ":", color=PALETTE[7], lw=2)
axes[1].plot(*zip((0, 0), shadow), color=PALETTE[4], lw=7, alpha=0.6, label="B's shadow on A")
axes[1].set_title(f"The dot product is length times shadow:\nA·B = {np.linalg.norm(A):.2f} × {np.linalg.norm(shadow):.2f} = {A @ B:.2f}")
arrow(axes[2], A, DISEASED, "A"); arrow(axes[2], C, PALETTE[4], "C = 0.4 × A", dy=-0.3)
axes[2].plot(*zip(A, C), "--", color=PALETTE[7], lw=2, label=f"gap between the tips = {np.linalg.norm(A - C):.2f}")
axes[2].set_title("Cosine ignores severity:\ncos(A, C) = 1.00 although C is far milder")
for ax in axes:
    ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
    ax.set(xlim=(-0.4, 2.7), ylim=(-0.4, 2.5), xlabel="bm_01 (SD units)", ylabel="bm_02 (SD units)", aspect="equal")
    ax.legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()

---
## 4. Implementation from Scratch

**Part 1** writes each operation as a loop over the tests, so that every step of the definition is visible. **Part 2** rewrites them as matrix products that compare whole groups of patients at once. **Part 3** adds rank and the seen/new split. The last cell shows what the cosine matrix looks like for real new patients.

In [ ]:
# NumPy implementation, heavily commented — part 1: one patient against another, written as loops to show the definitions
def dot(x, y):
    """Σ_j x_j·y_j: multiply matching tests, add up the products."""
    total = 0.0
    for xj, yj in zip(x, y):
        total += xj * yj
    return total

def norm(x):
    """Length of a patient vector: √(x·x) — Pythagoras in p dimensions."""
    return np.sqrt(dot(x, x))

def cosine(x, y):
    """cos θ = x·y / (‖x‖‖y‖): same pattern (+1), unrelated (0), opposite (−1)."""
    return dot(x, y) / (norm(x) * norm(y))

def euclidean(x, y):
    """Straight-line gap between the two arrow tips: ‖x − y‖."""
    return norm(x - y)

p0, p1 = X_all[0], X_all[1]
print(f"dot  (loop) {dot(p0, p1): .6f}   numpy @ {p0 @ p1: .6f}")
print(f"norm (loop) {norm(p0): .6f}   numpy   {np.linalg.norm(p0): .6f}")
print(f"cosine {cosine(p0, p1): .4f}   euclidean {euclidean(p0, p1): .4f}")
print(f"identity check: ‖x−y‖² = ‖x‖² + ‖y‖² − 2x·y → {euclidean(p0, p1)**2:.6f} = {norm(p0)**2 + norm(p1)**2 - 2 * dot(p0, p1):.6f}")

**Part 2 — the whole panel at once.** Looping over 600 new patients × 1,400 seen patients × 12 tests is 10 million Python steps. One matrix product does the same arithmetic in compiled code. Every function below is the loop version above, rewritten so that each row of a matrix is a patient. The standardiser learns each test's mean and SD from the **training patients only**: a new patient is put into units learned from patients we have already seen, never from themselves.

In [ ]:
# Part 2: matrix versions. Rows are patients, columns are tests.
def fit_standardiser(X):
    """Each test's mean and SD, learned from the rows given (the training patients)."""
    return X.mean(axis=0), X.std(axis=0)

def standardise(X, mu, sd):
    return (X - mu) / sd                       # broadcasting: (n, p) − (p,) is applied row by row

def gram(P, Q):
    """Every pairwise dot product: G[i, j] = P[i]·Q[j]. One matrix product, shape (n_P, n_Q)."""
    return P @ Q.T

def cosine_matrix(P, Q):
    """Shrink every row to length 1 first; then the dot products ARE the cosines."""
    Pn = P / np.sqrt((P ** 2).sum(axis=1, keepdims=True))
    Qn = Q / np.sqrt((Q ** 2).sum(axis=1, keepdims=True))
    return Pn @ Qn.T

def sq_distance_matrix(P, Q):
    """‖p − q‖² = ‖p‖² + ‖q‖² − 2 p·q for every pair, without a loop."""
    d2 = (P ** 2).sum(axis=1)[:, None] + (Q ** 2).sum(axis=1)[None, :] - 2 * P @ Q.T
    return np.maximum(d2, 0)                   # rounding can leave −1e−15; a squared distance is never negative

def knn_score(S, y_seen, k, larger_is_closer=True):
    """Risk for each new patient = share of diseased patients among its k most similar seen patients."""
    order = np.argsort(-S if larger_is_closer else S, axis=1)[:, :k]
    return y_seen[order].mean(axis=1)

**Part 3 — rank: how many tests are genuinely different?** Row-reduce the panel exactly as in school simultaneous equations: use one column to clear the entries below it, move to the next column, and count how many columns still have something left (a *pivot*). A test whose column is wiped out completely was a combination of the earlier tests: it adds no new direction. Then split the patients 70/30 into seen (training) and new (test), stratified on disease, and compute every new-against-seen cosine in one matrix product.

In [ ]:
# Part 3: rank by Gaussian elimination, then the seen/new split
def rank_by_elimination(M, tol=1e-10):
    """Number of independent columns: row-reduce and count pivots larger than tol × the largest entry."""
    M = M.astype(float).copy()
    n, p = M.shape
    tol = tol * np.abs(M).max()
    r = 0
    for j in range(p):                                     # walk across the tests
        piv = r + np.argmax(np.abs(M[r:, j]))              # partial pivoting: the largest remaining entry
        if np.abs(M[piv, j]) <= tol:
            continue                                       # nothing new here: a combination of earlier tests
        M[[r, piv]] = M[[piv, r]]                          # swap the pivot row up
        M[r + 1:] -= np.outer(M[r + 1:, j] / M[r, j], M[r])  # clear everything below the pivot
        r += 1
        if r == n:
            break
    return r

rng = np.random.default_rng(SEED)                          # stratified 70/30 split, no library
idx_te = np.sort(np.concatenate([rng.choice(np.where(y_all == c)[0], int(round(0.3 * (y_all == c).sum())), replace=False) for c in (0, 1)]))
idx_tr = np.setdiff1d(np.arange(len(y_all)), idx_te)
X_tr_raw, X_te_raw, y_tr, y_te = X_all[idx_tr], X_all[idx_te], y_all[idx_tr], y_all[idx_te]
mu, sd = fit_standardiser(X_tr_raw)                        # learned from seen patients only
X_tr, X_te = standardise(X_tr_raw, mu, sd), standardise(X_te_raw, mu, sd)

C_te_tr = cosine_matrix(X_te, X_tr)                        # (n_new, n_seen): every new patient against every seen one
risk_knn = knn_score(C_te_tr, y_tr, k=25)                  # k = 25 fixed in advance (≈ √n_seen / 1.5), not tuned
print(f"seen {len(y_tr)} (prev {y_tr.mean():.1%}) · new {len(y_te)} (prev {y_te.mean():.1%}) · cosine matrix {C_te_tr.shape}")
print(f"rank of the 12-test training panel by elimination: {rank_by_elimination(X_tr)}")
RESULTS = {"scratch cosine 25-NN, held-out AUC": roc_auc_score(y_te, risk_knn)}
print(f"held-out AUC of the from-scratch cosine 25-NN: {RESULTS['scratch cosine 25-NN, held-out AUC']:.3f}")

In [ ]:
# What the cosine matrix looks like: new patients against each other, sorted by disease status
C_te_te = cosine_matrix(X_te, X_te)
d_idx, h_idx = np.where(y_te == 1)[0], np.where(y_te == 0)[0]
off = ~np.eye(len(y_te), dtype=bool)                        # leave out each patient's cosine with itself (always 1)
blocks = {"diseased–diseased": C_te_te[np.ix_(d_idx, d_idx)][off[np.ix_(d_idx, d_idx)]],
          "diseased–healthy": C_te_te[np.ix_(d_idx, h_idx)].ravel(),
          "healthy–healthy": C_te_te[np.ix_(h_idx, h_idx)][off[np.ix_(h_idx, h_idx)]]}
RESULTS.update({f"mean cosine, {k} pairs": v.mean() for k, v in blocks.items()})
lengths = np.sqrt((X_tr ** 2).sum(axis=1))                  # how far each seen patient sits from the average patient
RESULTS.update({"mean arrow length, seen diseased": lengths[y_tr == 1].mean(), "mean arrow length, seen healthy": lengths[y_tr == 0].mean()})

sel = np.r_[d_idx[:25], h_idx[:25]]
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.15, 1]})
sns.heatmap(C_te_te[np.ix_(sel, sel)], cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, ax=ax[0],
            xticklabels=False, yticklabels=False, cbar_kws={"label": "cosine similarity"})
ax[0].axhline(25, color="k", lw=2); ax[0].axvline(25, color="k", lw=2)
ax[0].set_title("Diseased patients point the same way (warm top-left block);\nhealthy patients point in every direction")
ax[0].set(xlabel="25 diseased | 25 healthy new patients", ylabel="25 diseased | 25 healthy new patients")
parts = ax[1].violinplot(list(blocks.values()), showmeans=True)
for body, col in zip(parts["bodies"], [DISEASED, PALETTE[4], HEALTHY]):
    body.set_facecolor(col); body.set_alpha(0.55)
ax[1].set_xticks([1, 2, 3], list(blocks))
ax[1].axhline(0, color="k", lw=0.8, ls=":")
ax[1].set_title("Over all new pairs: diseased pairs share a direction;\nhealthy pairs average out near zero")
ax[1].set(ylabel="cosine similarity between two new patients")
plt.tight_layout(); plt.show()
print({k: round(v, 3) for k, v in RESULTS.items()})

---
## 5. Library Implementation

scikit-learn and NumPy's linear-algebra routines do the same jobs. The first cell checks that every scratch result matches the library one. The second asks the clinical question on the new patients: does "which seen patients does this one resemble?" predict disease? The yardstick is `datasets.baselines.clinician_baseline("screening", ...)`, a two-test rule that flags a patient if `bm_01` or `bm_02` is more than 1 SD above average. The vote is compared with the rule at the rule's own specificity, so neither gets to pick an easier operating point.

In [ ]:
# sklearn / torch / library version — the same operations, checked against the scratch versions
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.neighbors import KNeighborsClassifier

scaler = StandardScaler().fit(X_tr_raw)                     # the job fit_standardiser did
Xs_tr, Xs_te = scaler.transform(X_tr_raw), scaler.transform(X_te_raw)
knn = KNeighborsClassifier(n_neighbors=25, metric="cosine", algorithm="brute").fit(Xs_tr, y_tr)
risk_sk = knn.predict_proba(Xs_te)[:, 1]

checks = pd.Series({
    "standardised panel": np.allclose(Xs_te, X_te),
    "cosine matrix": np.allclose(cosine_similarity(X_te, X_tr), C_te_tr),
    "squared-distance matrix": np.allclose(euclidean_distances(X_te, X_tr) ** 2, sq_distance_matrix(X_te, X_tr)),
    "rank (SVD) = rank (elimination)": np.linalg.matrix_rank(X_tr) == rank_by_elimination(X_tr),
    "25-NN risk scores": np.allclose(risk_sk, risk_knn),
}, name="scratch = library")
print(checks.to_string())
print(f"largest risk-score disagreement: {np.abs(risk_sk - risk_knn).max():.3f}")

# The clinician yardstick: a two-test rule, positive if bm_01 or bm_02 is > 1 SD (age adds a nudge)
rule = clinician_baseline("screening", df.iloc[idx_te]) >= 0.5
sens_rule, spec_rule = rule[y_te == 1].mean(), 1 - rule[y_te == 0].mean()
fpr, tpr, thr = roc_curve(y_te, risk_sk)
j = np.where(fpr <= 1 - spec_rule)[0][-1]                   # the kNN operating point at the rule's specificity
RESULTS.update({"library cosine 25-NN, held-out AUC": roc_auc_score(y_te, risk_sk),
                "clinician rule sensitivity": sens_rule, "clinician rule specificity": spec_rule,
                "25-NN sensitivity at the rule's specificity": tpr[j]})
print(pd.Series(RESULTS).round(3).to_string())

In [ ]:
# Does "patients like this one" predict disease in new patients? Held-out patients only.
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].plot(fpr, tpr, lw=2.5, color=PALETTE[2], label=f"cosine 25-NN (AUC {RESULTS['library cosine 25-NN, held-out AUC']:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.3, label="chance")
ax[0].scatter([1 - spec_rule], [sens_rule], s=160, marker="s", color=PALETTE[8], zorder=5,
              label=f"clinician two-test rule (sens {sens_rule:.2f}, spec {spec_rule:.2f})")
ax[0].scatter([fpr[j]], [tpr[j]], s=160, color=PALETTE[2], edgecolor="k", zorder=5,
              label=f"25-NN at the same specificity (sens {tpr[j]:.2f})")
ax[0].set_title("At the rule's own specificity the neighbour vote catches about as many\ndiseased patients, and its threshold can slide along the whole curve")
ax[0].set(xlabel="1 − specificity (healthy patients wrongly flagged)", ylabel="sensitivity (diseased patients caught)")
ax[0].legend(loc="lower right", fontsize=9)
bins = np.linspace(0, 1, 26)
for d, lab, col in [(0, "healthy", HEALTHY), (1, "diseased", DISEASED)]:
    ax[1].hist(risk_sk[y_te == d], bins=bins, density=True, alpha=0.6, color=col, label=f"{lab} new patients")
ax[1].set_title("Most healthy patients have few diseased neighbours;\nmost diseased patients have many")
ax[1].set(xlabel="share of the 25 most similar seen patients who were diseased", ylabel="density")
ax[1].legend()
plt.tight_layout(); plt.show()

---
## 6. Experiments & Variations

The three experiments are the manifest's pre-registered list, in that order. Each is a sweep or a comparison on the held-out (new) patients, and each is checked against what the generator built in (`truth`).

### Experiment 1: cosine similarity between patient vectors

**Design.** Two comparisons on the same new patients. (a) Score each new patient by their cosine with the *typical diseased profile* — the average standardised panel of the seen diseased patients, $\mu_D$ — and compare it with the plain dot product $x\cdot\mu_D$, which keeps severity. (b) Let the $k$ most similar seen patients vote, with $k$ from 1 to 201, under three similarity measures: cosine, Euclidean distance and the raw dot product. The $k$ sweep is read as a curve. The $k = 25$ used in section 5 was fixed before this sweep and is not re-chosen from it. **Truth check:** the largest entries of $\mu_D$ should be the tests the generator made strong (`bm_01`, `bm_02`), plus `bm_12`, which is `bm_01`'s near-copy.

In [ ]:
# CENTRAL
# VISUAL 3 — Experiment 1: cosine similarity between patient vectors
mu_D = X_tr[y_tr == 1].mean(axis=0)                    # the typical diseased profile, from seen patients only
print("largest entries of the diseased profile:", pd.Series(mu_D, index=BM).nlargest(4).round(2).to_dict())
print("truth — strong:", truth["strong"], "· collinear pair:", truth["collinear"])
cos_to_D = cosine_matrix(X_te, mu_D[None, :]).ravel()  # direction only
dot_to_D = X_te @ mu_D                                 # matrix–vector product: direction AND severity, one linear score

ks = [1, 3, 5, 11, 25, 51, 101, 201]
measures = {"cosine": (C_te_tr, True), "Euclidean distance": (sq_distance_matrix(X_te, X_tr), False),
            "raw dot product": (gram(X_te, X_tr), True)}
auc_k = pd.DataFrame({m: [roc_auc_score(y_te, knn_score(S, y_tr, k, big)) for k in ks]
                      for m, (S, big) in measures.items()}, index=pd.Index(ks, name="k"))
RESULTS.update({"cosine with diseased profile, AUC": roc_auc_score(y_te, cos_to_D),
                "dot product with diseased profile, AUC": roc_auc_score(y_te, dot_to_D)})
RESULTS.update({f"{m} 1-NN, AUC": auc_k.loc[1, m] for m in auc_k})

fig, ax = plt.subplots(1, 2, figsize=(16, 6.2))
fig.suptitle("New patients who point the same way as the typical diseased patient are usually diseased", fontweight="bold")
for d, lab, col in [(0, "healthy", HEALTHY), (1, "diseased", DISEASED)]:
    ax[0].hist(cos_to_D[y_te == d], bins=np.linspace(-1, 1, 41), density=True, alpha=0.6, color=col, label=f"{lab} new patients")
ax[0].set_title(f"Direction alone separates the groups: cosine AUC {RESULTS['cosine with diseased profile, AUC']:.3f},\n"
                f"barely below the severity-keeping dot product ({RESULTS['dot product with diseased profile, AUC']:.3f})", fontsize=12)
ax[0].set(xlabel="cosine similarity with the typical diseased profile", ylabel="density"); ax[0].legend(loc="upper left")
for c, m in zip([PALETTE[2], PALETTE[3], PALETTE[4]], auc_k):
    ax[1].plot(ks, auc_k[m], "o-", lw=2.5, color=c, label=m)
ax[1].set_xscale("log"); ax[1].minorticks_off(); ax[1].set_xticks(ks, ks)
ax[1].set_title("Neighbour vote: the three measures agree within about 0.01 from k = 11;\nat k = 1 the raw dot product leads, because it leans on long arrows", fontsize=12)
ax[1].set(xlabel="k, seen patients consulted (log scale)", ylabel="held-out AUC"); ax[1].legend()
plt.tight_layout(); plt.show()
print(auc_k.round(3).T.to_string())

🗣️ The typical diseased patient's arrow points mostly along `bm_01`, its near-copy `bm_12`, and `bm_02` (1.12, 1.12 and 0.87 SD above average). Those are the generator's two strong tests and the copy of one of them. New patients whose arrows point that way are usually diseased: the direction alone ranks them with an AUC of 0.927. Adding back how *far* they point, using the dot product, barely changes this (0.931). When the most similar seen patients vote instead, the three ways of measuring similarity give the same answer once 11 or more neighbours vote (AUC 0.90–0.93). With a single neighbour every measure is worse (0.73–0.83), because one look-alike patient is a noisy witness.

💻 `cosine_matrix(X_te, mu_D[None, :])` is one matrix–vector product after normalising, and `X_te @ mu_D` is the same product without normalising: a linear risk score in one line. The k sweep reuses three precomputed 600 × 1,400 similarity matrices, so each extra value of k costs only a sort.

🔬 For a fixed new patient $x$, the dot product ranks seen patients by $\langle x, y\rangle = \|x\|\,\|y\|\cos\theta$. Since $\|x\|$ is constant, that is a ranking by $\|y\|\cos\theta$, which favours long seen arrows. Seen diseased patients are longer on average (mean length 3.78 against 3.31 for healthy), because disease shifts their mean away from the seen-patient average. So for a new patient pointing the diseased way, the dot product's single "nearest" patient is more often a clearly diseased one. That is the likely reason it leads at $k = 1$ (0.826 against 0.732 for cosine and 0.735 for Euclidean). On unit vectors the cosine and Euclidean rankings are identical ($\|x - y\|^2 = 2 - 2\cos\theta$). Without normalisation they differ only through arrow length, and their AUCs stay within 0.02 of each other at every k. Differences that small are inside the bootstrap range of a single AUC on 92 diseased new patients (section 8: 0.88–0.95), so this sweep does not rank the three measures.

### Experiment 2: scaling the panel changes the geometry

**Design.** Add age to the panel, so each patient is 13 numbers: age plus the 12 tests. Compare two versions of the same patients. **As recorded:** age in years, with an SD of about 14, next to tests with an SD of about 1. **Standardised:** every column divided by its seen-patient SD. Then sweep a weight $w$ from 0.1 to 100 on one column at a time, which is the diagonal rescaling $x \mapsto Dx$. The two columns swept are:
- **age**, which carries no disease signal by construction (`load_screening` draws disease without reference to age; age only nudges `bm_02`);
- **`bm_01`**, the strongest signal.

The measures are the held-out AUC of a Euclidean 25-NN vote and, for one index patient, which seen patients count as their 25 nearest neighbours.

In [ ]:
# Experiment 2: scaling the panel changes the geometry — the sweep (figure in the next cell)
def nn_euclid(P_seen, P_new, k=25):
    """Indices of each new patient's k nearest seen patients (Euclidean)."""
    return np.argsort(sq_distance_matrix(P_new, P_seen), axis=1)[:, :k]

def auc_euclid(P_seen, P_new, k=25):
    return roc_auc_score(y_te, y_tr[nn_euclid(P_seen, P_new, k)].mean(axis=1))

age = df["age"].to_numpy()
rec_tr, rec_te = np.c_[age[idx_tr], X_tr_raw], np.c_[age[idx_te], X_te_raw]   # as recorded: years + SD units
m13, s13 = fit_standardiser(rec_tr)
std_tr, std_te = standardise(rec_tr, m13, s13), standardise(rec_te, m13, s13)

def reweight(col, w):
    D = np.ones(13); D[col] = w                         # x → Dx: stretch one test's axis by w
    return std_tr * D, std_te * D

weights = np.logspace(-1, 2, 13)
sweep = pd.DataFrame({"age (no disease signal)": [auc_euclid(*reweight(0, w)) for w in weights],
                      "bm_01 (strongest signal)": [auc_euclid(*reweight(1, w)) for w in weights]}, index=weights)
nn_rec, nn_std = nn_euclid(rec_tr, rec_te), nn_euclid(std_tr, std_te)
overlap = np.mean([len(set(a) & set(b)) / 25 for a, b in zip(nn_rec, nn_std)])
age_w_recorded = s13[0] / np.median(s13[1:])            # how much heavier age is when left in years
RESULTS.update({"25-NN AUC, as recorded (age in years)": auc_euclid(rec_tr, rec_te),
                "25-NN AUC, standardised": auc_euclid(std_tr, std_te),
                "neighbours shared, recorded vs standardised": overlap})
i0 = np.where(y_te == 1)[0][np.argmin(np.abs(X_te_raw[y_te == 1, 0] - 1.6))]   # a typical diseased new patient
print(f"age weight implied by recording in years ≈ {age_w_recorded:.1f}; neighbours shared on average {overlap:.0%}")
print(sweep.round(3).T.to_string())

In [ ]:
# VISUAL 4 — Experiment 2: scaling the panel changes the geometry (figure)
fig, axes = grid(3, ncols=3, panel=(6.2, 5.6))
for ax, nn in [(axes[0], nn_rec[i0]), (axes[1], nn_std[i0])]:
    ax.scatter(rec_tr[:, 0], rec_tr[:, 1], s=6, color=PALETTE[7], alpha=0.25, label="seen patients")
    ax.scatter(rec_tr[nn, 0], rec_tr[nn, 1], s=40, c=[DISEASED if y else HEALTHY for y in y_tr[nn]], edgecolor="k", lw=0.4,
               label=f"25 nearest: {y_tr[nn].sum()} diseased (orange)")
    ax.scatter(rec_te[i0, 0], rec_te[i0, 1], s=320, marker="*", color=PALETTE[6], edgecolor="k", label="index patient (diseased)")
    ax.set(xlabel="age (years)", ylabel="bm_01 (SD units)"); ax.legend(loc="lower right", fontsize=8)
axes[0].set_title("As recorded, neighbours are age-matched\nand scattered on the disease marker")
axes[1].set_title("Standardised, neighbours are matched\non the disease marker across ages")
for col, c in zip(sweep, [PALETTE[4], PALETTE[2]]):
    axes[2].plot(weights, sweep[col], "o-", lw=2.5, color=c, label=f"stretch {col}")
axes[2].axvline(1, color="k", ls=":", label="standardised (w = 1)")
axes[2].scatter([age_w_recorded], [RESULTS["25-NN AUC, as recorded (age in years)"]], s=160, marker="s", color=PALETTE[8], zorder=5,
                label=f"age left in years (w ≈ {age_w_recorded:.0f})")
axes[2].set_xscale("log")
axes[2].set_title("Units are hidden weights: stretching age costs up to\n0.2 of AUC; stretching the best test never helps")
axes[2].set(xlabel="weight w on one column (log scale)", ylabel="held-out AUC, Euclidean 25-NN"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

🗣️ Age is recorded in years, with a spread of about 14. Every test is in SD units, with a spread of about 1. So a 10-year age gap counts as much as a 10-SD gap on a test, and age outvotes the whole panel. The index patient (66 years old, `bm_01` = 1.6) shows what that does:
- **Age left in years:** all 25 "most similar" seen patients are aged 64–68, but their `bm_01` results run from −0.1 to 3.5, and only 7 of them were diseased.
- **Standardised:** the neighbours range from about 48 to 83 years old but cluster on the disease marker, and 10 were diseased.

Across all 600 new patients, the two versions share only 21% of their neighbours, and the AUC falls from 0.924 to 0.898 when age is left in years. Stretching age further (×100) drags the AUC down to 0.72. Stretching `bm_01`, the best test, never beats plain standardisation (0.924 at w = 1, 0.892 at w = 100).

💻 Learn the scaling on the seen patients and apply it to both groups: `StandardScaler().fit(X_seen)`, then `.transform` both. Better still, use `make_pipeline(StandardScaler(), KNeighborsClassifier(...))` so that cross-validation refits the scaler in every fold. A per-column weight is one broadcast multiply, `X * w`.

🔬 Multiplying column $j$ by $w$ replaces $\|x - y\|^2 = \sum_i (x_i - y_i)^2$ with $\sum_{i \ne j} (x_i - y_i)^2 + w^2 (x_j - y_j)^2$. As $w \to \infty$ the neighbour ranking tends to a ranking on column $j$ alone; as $w \to 0$ it ignores column $j$. Standardising is the choice $w_j = 1/\mathrm{SD}_j$, which gives every column an equal share of variance. That is a convention, not an optimum. It assumes each test deserves an equal say, and Experiment 3 shows a duplicated test quietly breaking that assumption. Because age carries no disease signal here, up-weighting it mixes uninformative variation into the ranking. Up-weighting `bm_01` converges on single-test performance, which is lower than the whole panel's. Neither beat the equal-variance geometry on this generator. That is a property of this generator, not a general law.

### Experiment 3: rank-deficient panels (redundant tests)

**Design.** First, find the redundancy without being told: take the singular values of the standardised 12-test panel and the direction belonging to the smallest one. **Truth check:** it should load on the pair the generator made collinear (`bm_12` ≈ `bm_01`). Then sweep: rebuild `bm_12` as `bm_01` + σ·noise, with σ falling from 1 to exactly 0, the point where the second test is a perfect copy. At each σ, fit a linear score on the seen patients two ways. **Normal equations** solve $(X^\top X)w = X^\top y$ directly. **Least squares via the SVD** (`np.linalg.lstsq`) returns the shortest of the equally good answers. Record the rank, the condition number, the two coefficients and the held-out AUC of the score $Xw$. Finally, ask whether the duplicate distorts *similarity*: compare the held-out 25-NN AUC with and without `bm_12`.

In [ ]:
# Experiment 3: rank-deficient panels (redundant tests) — the sweep (figure in the next cell)
U, sv, Vt = np.linalg.svd(X_tr, full_matrices=False)
v_min = pd.Series(Vt[-1], index=BM)                     # the direction the panel barely uses
print(f"singular values {sv.round(1)}; condition number {sv[0] / sv[-1]:.1f}")
print("smallest direction loads on:", v_min.abs().nlargest(2).round(3).to_dict(), "· truth collinear pair:", truth["collinear"])

sigmas = [1.0, 0.3, 0.1, 0.03, 1e-2, 1e-3, 1e-4, 1e-6, 1e-8, 0.0]
rng = np.random.default_rng(SEED)
e_tr, e_te = rng.normal(size=len(y_tr)), rng.normal(size=len(y_te))
rows = []
for sg in sigmas:
    A_tr, A_te = X_tr.copy(), X_te.copy()
    A_tr[:, 11], A_te[:, 11] = X_tr[:, 0] + sg * e_tr, X_te[:, 0] + sg * e_te
    A1_tr, A1_te = np.c_[np.ones(len(A_tr)), A_tr], np.c_[np.ones(len(A_te)), A_te]   # leading column of ones = intercept
    try:
        w_ne = np.linalg.solve(A1_tr.T @ A1_tr, A1_tr.T @ y_tr)   # normal equations: needs XᵀX invertible
    except np.linalg.LinAlgError:
        w_ne = np.full(A1_tr.shape[1], np.nan)
    w_ls = np.linalg.lstsq(A1_tr, y_tr, rcond=None)[0]           # SVD route: shortest w among equal fits
    rows.append(dict(sigma=sg, rank=np.linalg.matrix_rank(A_tr), cond=np.linalg.cond(A_tr),
                     w01_ne=w_ne[1], w12_ne=w_ne[12], w01_ls=w_ls[1], w12_ls=w_ls[12],
                     auc_ne=roc_auc_score(y_te, A1_te @ w_ne) if np.isfinite(w_ne).all() else np.nan,
                     auc_ls=roc_auc_score(y_te, A1_te @ w_ls)))
exp3 = pd.DataFrame(rows)
keep = [i for i in range(12) if i != 11]
RESULTS.update({"panel condition number (12 tests)": sv[0] / sv[-1],
                "cosine 25-NN AUC with bm_12": RESULTS["scratch cosine 25-NN, held-out AUC"],
                "cosine 25-NN AUC without bm_12": roc_auc_score(y_te, knn_score(cosine_matrix(X_te[:, keep], X_tr[:, keep]), y_tr, 25))})
print(exp3.to_string(float_format=lambda v: f"{v:.3g}"))

In [ ]:
# VISUAL 5 — Experiment 3: rank-deficient panels (redundant tests) (figure)
xs, xlab = np.arange(len(sigmas)), ["1", "0.3", "0.1", "0.03", "0.01", "1e-3", "1e-4", "1e-6", "1e-8", "0\n(copy)"]
coef_sum = (exp3["w01_ls"] + exp3["w12_ls"]).agg(["min", "max"])
fig, axes = grid(4, ncols=2, panel=(8, 5))
axes[0].bar(range(1, 13), sv, color=[PALETTE[7]] * 11 + [DISEASED])
axes[0].set_yscale("log"); axes[0].set_xticks(range(1, 13))
axes[0].set_title(f"One direction is nearly empty: the smallest singular value\nis the bm_12 − bm_01 direction (condition number {sv[0] / sv[-1]:.0f})")
axes[0].set(xlabel="singular value, largest to smallest", ylabel="size (log scale)")
axes[1].plot(xs, exp3["cond"].replace(np.inf, np.nan), "o-", lw=2.5, color=PALETTE[3])
axes[1].set_yscale("log"); axes[1].set_xticks(xs, xlab)
axes[1].set_title("As the second test becomes a copy, the condition\nnumber climbs tenfold per tenfold drop in noise")
axes[1].set(xlabel="noise σ separating bm_12 from bm_01", ylabel="condition number of the panel (log)")
for col, lab, c, ls in [("w01_ls", "bm_01, SVD least squares", PALETTE[2], "-"), ("w12_ls", "bm_12, SVD least squares", PALETTE[4], "-"),
                        ("w01_ne", "bm_01, normal equations", PALETTE[2], "--"), ("w12_ne", "bm_12, normal equations", PALETTE[4], "--")]:
    axes[2].plot(xs, exp3[col], "o" + ls, lw=2, color=c, label=lab)
axes[2].set_yscale("symlog", linthresh=0.1); axes[2].set_xticks(xs, xlab)
axes[2].set_title(f"The two coefficients run off in opposite directions;\nonly their sum is pinned down ({coef_sum['min']:.3f} to {coef_sum['max']:.3f} at every σ)")
axes[2].set(xlabel="noise σ separating bm_12 from bm_01", ylabel="coefficient (symmetric log)"); axes[2].legend(fontsize=8)
axes[3].plot(xs, exp3["auc_ls"], "o-", lw=2.5, color=PALETTE[2], label="SVD least squares")
axes[3].plot(xs, exp3["auc_ne"], "s--", lw=2.5, color=PALETTE[3], label="normal equations")
axes[3].set_xticks(xs, xlab); axes[3].set_ylim(0.5, 1.0)
axes[3].set_title("Yet the held-out AUC of the score does not move:\nprediction survives what interpretation does not")
axes[3].set(xlabel="noise σ separating bm_12 from bm_01", ylabel="held-out AUC of the linear score Xw"); axes[3].legend(loc="lower left")
plt.tight_layout(); plt.show()
print({k: round(RESULTS[k], 3) for k in ["cosine 25-NN AUC with bm_12", "cosine 25-NN AUC without bm_12"]})

🗣️ Two of the twelve tests are nearly the same test: `bm_12` is `bm_01` plus a little noise, so ordering both tells you barely more than ordering one. Linear algebra finds this without being told. The panel has twelve directions, but one is almost empty: its size is 2.3, against 34–54 for the other eleven. That direction is "`bm_12` minus `bm_01`", the pair the generator made near-copies. As the two tests become exact copies, a model fitted to the panel can no longer tell how to share the credit between them. It gives one a huge positive weight and the other a huge negative one, ±90,000 when the noise is 10⁻⁸. Only their total, 0.144, is fixed by the data. Yet the *predictions* do not move: the held-out AUC is 0.935 at every step. A redundant test ruins the explanation, not the prediction. For similarity it quietly counts `bm_01` twice. Here that happened to cost nothing (25-NN AUC 0.919 with the copy, 0.915 without), because the test counted twice was the best one.

💻 Check `np.linalg.matrix_rank(X)` and `np.linalg.cond(X)` before fitting anything. Solve least squares with `np.linalg.lstsq` (SVD) or a QR routine, not by forming $X^\top X$. Note that `np.linalg.solve` did **not** raise an error at σ = 0. It silently returned one of infinitely many answers, (0.020, 0.125).

🔬 With columns $a$ and $a + \sigma e$, the smallest singular value of $X$ shrinks in proportion to $\sigma$, so $\kappa(X) \propto 1/\sigma$: 206 at σ = 0.01, 2,064 at 10⁻³ and 2.06 × 10⁸ at 10⁻⁸, tenfold per tenfold. The data pin down $w_{01} + w_{12}$, which is 0.144 at every σ. The split between the two is set only by the tiny $\sigma e$ component, so each coefficient grows like $1/\sigma$: about 9 at 10⁻⁴, 905 at 10⁻⁶ and 90,480 at 10⁻⁸. The normal equations work with $X^\top X$, whose condition number is $\kappa^2 \approx 4 \times 10^{16}$ at σ = 10⁻⁸. That is beyond double precision (machine ε ≈ 2.2 × 10⁻¹⁶), and they return 9,199 where the SVD route gives 90,480, wrong by a factor of ten. At σ = 0 the rank drops to 11. The solutions then form a line $w^\ast + t\,(1, -1)$ in the $(w_{01}, w_{12})$ plane, and `lstsq` picks its minimum-norm point, (0.072, 0.072). Predictions $Xw$ are identical everywhere on that line, which is why the AUC is flat. Under this notebook's `predict` goal, these coefficients are not mechanisms at any σ.

---
## 7. Failure Modes & Edge Cases

Two deliberate breaks, both on one draw from the same generator:
1. **Pad the panel.** Ask `load_screening` for 312 tests instead of 12 and add them in nested steps, so that each wider panel contains the narrower one. Every test beyond the twelfth carries no signal by construction (`truth`).
2. **Watch distances concentrate.** For each new patient, compare the distance to the nearest seen patient with the distance to the farthest. If the two are almost the same, "nearest" has stopped meaning anything.

The reference line is the vote using only the five tests the generator made informative.

In [ ]:
# VISUAL — deliberately break the method; show what bad looks like and why
wide, truth_w = load_screening(n_biomarkers=312, return_truth=True)   # one draw: the usual 12 tests + 300 that carry no signal
assert (wide["disease"].to_numpy() == y_all).all()                  # same patients' disease status, so the same split applies
W = wide.filter(like="bm_").to_numpy()
extra = [0, 4, 12, 36, 100, 300]
rows = []
for m in extra:                                                      # nested panels: first 12 tests, then 12 + m
    mu_m, sd_m = fit_standardiser(W[idx_tr, :12 + m])
    P_tr, P_te = standardise(W[idx_tr, :12 + m], mu_m, sd_m), standardise(W[idx_te, :12 + m], mu_m, sd_m)
    D2 = sq_distance_matrix(P_te, P_tr)
    near, far = np.sqrt(D2.min(axis=1)), np.sqrt(D2.max(axis=1))
    rows.append(dict(n_tests=12 + m, contrast=np.mean((far - near) / near),
                     auc_cosine=roc_auc_score(y_te, knn_score(cosine_matrix(P_te, P_tr), y_tr, 25)),
                     auc_euclid=roc_auc_score(y_te, knn_score(D2, y_tr, 25, larger_is_closer=False))))
fail = pd.DataFrame(rows)
inf_cols = [BM.index(c) for c in truth_w["strong"] + truth_w["weak"]]   # the five tests the generator made informative
mu5, sd5 = fit_standardiser(W[idx_tr][:, inf_cols])
auc_truth5 = roc_auc_score(y_te, knn_score(cosine_matrix(standardise(W[idx_te][:, inf_cols], mu5, sd5), standardise(W[idx_tr][:, inf_cols], mu5, sd5)), y_tr, 25))
RESULTS.update({"failure: cosine 25-NN AUC, 12 tests": fail["auc_cosine"].iloc[0],
                "failure: cosine 25-NN AUC, 312 tests (300 noise)": fail["auc_cosine"].iloc[-1],
                "failure: Euclidean 25-NN AUC, 312 tests (300 noise)": fail["auc_euclid"].iloc[-1],
                "failure: cosine 25-NN AUC, 5 truth-informative tests only": auc_truth5})

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].plot(fail["n_tests"], fail["auc_cosine"], "o-", lw=2.5, color=PALETTE[2], label="cosine 25-NN")
ax[0].plot(fail["n_tests"], fail["auc_euclid"], "s--", lw=2.5, color=PALETTE[3], label="Euclidean 25-NN")
ax[0].axhline(auc_truth5, color=PALETTE[8], ls=":", label=f"only the 5 informative tests (truth): {auc_truth5:.3f}")
ax[0].axhline(0.5, color="k", alpha=0.3, label="chance")
ax[0].set_xscale("log"); ax[0].minorticks_off(); ax[0].set_xticks(fail["n_tests"], fail["n_tests"])
ax[0].set_title("Padding the panel with uninformative tests dilutes similarity:\nthe neighbour vote slides from its best towards chance")
ax[0].set(xlabel="tests in the panel (log scale; all beyond the first 12 are noise)", ylabel="held-out AUC"); ax[0].legend(fontsize=9)
ax[1].plot(fail["n_tests"], fail["contrast"], "o-", lw=2.5, color=PALETTE[4])
ax[1].set_xscale("log"); ax[1].minorticks_off(); ax[1].set_xticks(fail["n_tests"], fail["n_tests"])
ax[1].set_title("In a wide panel the nearest and farthest patients are almost\nthe same distance away: nearest stops meaning similar")
ax[1].set(xlabel="tests in the panel (log scale)", ylabel="(farthest − nearest) / nearest distance")
plt.tight_layout(); plt.show()
print(fail.round(3).to_string(index=False))

**What bad looks like.** With the usual 12 tests the cosine vote scores 0.930. With 300 uninformative tests added it scores 0.792, and the Euclidean vote falls from 0.916 to 0.735. Nothing errors and no warning appears: the method still returns a confident-looking risk for every patient. Meanwhile the gap between the nearest and the farthest seen patient shrinks from 3.1 times the nearest distance to 0.26 times.

**Why.** Each uninformative test adds an independent squared difference to every distance, of the same expected size for every pair of patients. The disease signal lives in about five directions, so its share of each distance shrinks as tests are added, and a patient's "nearest" neighbours drift towards random patients. Cosine degrades more slowly than Euclidean distance here (0.79 vs 0.74 at 312 tests), but both degrade. Using only the five truth-informative tests gives 0.933, the same as the full 12-test panel (0.930). Six no-signal tests and a duplicate cost little; 300 cost a lot.

**Other ways this breaks (not plotted):**
- **Leakage.** If the mean and SD are learned from all patients, new ones included, the new patients have helped set their own units. Fit on the seen patients only.
- **The zero vector.** A patient exactly at the seen-patient average has length 0, and their cosine is 0/0. Rounded or mean-imputed panels produce such patients.
- **Uncentred positive values.** Laboratory results in natural units are positive and mostly sit far from zero relative to their spread (sodium near 140 mmol/L). Without centring, every arrow points into the same corner and every cosine sits near 1. Cosine needs the origin to mean "average patient".
- **Missing results.** A missing test has no position on its axis. Dropping it shortens the vector, and filling it with the mean pulls the patient towards average.

---
## 8. Results, Findings & Implications

### 📊 Results

All results are on the 600 held-out patients (92 diseased, 508 healthy) unless a row says otherwise. The full table of every quoted number is computed in the next cell.

| Where | Comparison | Result |
|---|---|---|
| §5 | cosine 25-NN vote vs clinician two-test rule | AUC 0.919 (bootstrap 95% range 0.88–0.95); sensitivity 0.93 vs 0.90 at the rule's specificity of 0.67 |
| Exp 1 | cosine vs dot product with the typical diseased profile | AUC 0.927 vs 0.931 |
| Exp 1 | 1-NN vote: cosine / Euclidean / raw dot product | AUC 0.732 / 0.735 / 0.826; all three between 0.90 and 0.93 for k ≥ 11 |
| Exp 2 | age left in years vs standardised (Euclidean 25-NN) | AUC 0.898 vs 0.924; 21% of neighbours shared; age × 100 gives 0.724 |
| Exp 3 | near-copy noise σ from 1 to 0 | condition number 2.7 → 1.4 × 10¹⁶; coefficients ±90,480 at σ = 10⁻⁸; AUC 0.935 at every σ |
| §7 | 12 → 312 tests (300 uninformative) | cosine 0.930 → 0.792; Euclidean 0.916 → 0.735; five truth-informative tests alone 0.933 |

In [ ]:
# How far would the headline AUC move with a different 600 new patients? Resample the test set 1,000 times.
rng = np.random.default_rng(SEED)
boot = []
for _ in range(1000):
    b = rng.integers(0, len(y_te), len(y_te))
    boot.append(roc_auc_score(y_te[b], risk_sk[b]))
lo, hi = np.percentile(boot, [2.5, 97.5])
RESULTS.update({"cosine 25-NN AUC, bootstrap 2.5th percentile": lo, "cosine 25-NN AUC, bootstrap 97.5th percentile": hi})

# Every number quoted in Findings, gathered from the cells above (held-out patients unless the row says otherwise)
pd.Series(RESULTS, name="value").round(3).to_frame()

### 🔍 Findings

1. **Similarity predicts disease in new patients, within limits.** The score is the share of the 25 most similar seen patients who were diseased. It reaches an AUC of 0.919, and the bootstrap range is 0.88–0.95, so a single AUC here is uncertain to about ±0.03. At the clinician rule's specificity (0.67) it catches 0.93 of diseased patients against the rule's 0.90. That is about three more of 92: a match, not a demonstrated improvement. Its practical gain is that its threshold can move.
2. **Direction carries most of the signal.** Cosine with the typical diseased profile (direction only) scores 0.927; the dot product (direction and severity) scores 0.931. On this panel, severity adds little once direction is known. The profile's largest entries were `bm_01`, `bm_12` and `bm_02`, exactly the generator's strong tests plus the near-copy.
3. **The three similarity measures agree once enough neighbours vote.** For k ≥ 11 the cosine, Euclidean and raw dot-product votes stay within about 0.01 of each other, well inside the bootstrap range. With a single neighbour all three are poor, the dot product least so (0.83).
4. **Units are a hidden weighting.** Leaving age in years replaces 79% of each patient's neighbours and costs 0.025 of AUC. Pushing the age weight further costs up to 0.20. No single-column reweighting beat equal-variance standardisation on this generator.
5. **Redundancy wrecks coefficients and spares predictions.** The smallest singular direction alone identified the duplicated pair (loadings ±0.707 on `bm_01` and `bm_12`, matching the generator's truth). As the copy became exact, the coefficients ran to ±90,000 while the AUC stayed at 0.935. The normal equations went wrong first, and silently.
6. **Surprise:** up-weighting the strongest test never helped, and the five truth-informative tests alone scored the same as the full panel (0.933 vs 0.930). A few uninformative tests are cheap; 300 of them cost 0.14 of AUC.

### 💡 Implications

- **Use** dot products, cosines and distances when a decision rests on "patients like this one": nearest-neighbour risk, case retrieval, triage by resemblance. Use the matrix–vector product $Xw$ whenever a score is a weighted sum of results.
- **Prefer cosine** when severity should not count ("the same pattern, any degree"). **Prefer distance or the dot product** when severity is the point.
- **Standardise first, on the seen patients only**, and treat the choice of units as a modelling decision. Write the weighting down.
- **Check rank and condition number before reading any coefficient.** If κ is large, report predictions, not weights, and solve with an SVD or QR routine, never the normal equations.
- **Avoid** raw similarity on wide panels of mostly uninformative tests. Select tests first, from domain knowledge or with the selection methods of later modules, or the "nearest" patients become arbitrary.

### ⚠️ Considerations

- **Assumptions**:
  - equal numeric gaps on a test mean equal clinical difference;
  - the seen-patient mean is a meaningful origin (for cosine);
  - results are present, not missing in a pattern;
  - patients with similar panels share disease risk.
- **Computational cost**:
  - a single dot product is O(p);
  - all pairs cost O(n_new · n_seen · p) time and O(n_new · n_seen) memory (6.7 MB here, 80 GB at 100,000 × 100,000);
  - rank and condition number via the SVD cost O(n · p²).
- **Scaling**: brute-force all-pairs comparison is fine up to roughly 10⁴ patients. Beyond that, compute in row blocks or use tree or approximate nearest-neighbour indices. Width is a worse enemy than length: similarity degrades as uninformative tests are added (section 7).
- **Common mistakes**:
  - standardising with all patients (leakage);
  - mixing units, such as years beside SD units;
  - including the same test twice under two names;
  - reading coefficients off an ill-conditioned panel;
  - taking cosines of uncentred positive lab values;
  - solving the normal equations.
- **Ethics / safety**: "patients like this one" inherits whoever is in the seen set. If a group is under-represented, its members' nearest neighbours are mostly other kinds of patient. Similarity also carries forward past patients' labels, including any past misdiagnoses. Every number here comes from synthetic patients with a known generator.

---
## 9. Clinical Reading

_Does this person have the condition? How sure are we?_

**What this means in the clinic.** A new patient's screening panel can be compared with every patient the service has already seen, by asking which of them point the same way. On these synthetic patients, a randomly chosen diseased patient outscores a randomly chosen healthy one about 92 times in 100 (AUC 0.92, bootstrap range 0.88–0.95). At the specificity of the two-test rule a clinician might use (0.67), it catches about as many diseased patients (0.93 vs 0.90). "How sure" comes from the vote itself. A score of 20/25 means that 20 of the 25 most similar past patients had the condition. That is a statement about past look-alikes, not a calibrated probability for this person.

**What a clinician would do differently having seen this:**
- Ask *what units the similarity was measured in* before trusting any "patients like this one" tool. If age sat in years beside tests in SD units, the tool was mostly matching on age: 79% of the neighbours change once it is standardised.
- Treat two near-identical tests on a panel as one test counted twice, and ask the tool's builders whether they checked rank and condition number.
- Prefer a short panel of tests known to be informative over a long panel of everything. Adding uninformative tests made the look-alikes less alike, with no warning.

**What they must not conclude:**
- That high similarity to diseased patients means this patient has the disease. It means they resemble past diseased patients on these twelve tests: a prediction to act on with the usual confirmatory work-up, not a diagnosis.
- That a test with a large weight in a linear score is a cause, or even the most important test. Experiment 3 produced weights of ±90,000 with unchanged predictions. Under this notebook's `predict` goal, weights are not mechanisms.
- That sensitivity of 0.93 against 0.90 shows the vote is better than the rule. The difference is about three patients out of 92, well inside the spread seen on resampling.
- That any of these numbers transfer to a real service. These are **retrospective results on synthetic patients**, from a generator built to contain two strong tests and a duplicate. The bar for clinical use is a prospective evaluation on the service's own patients, with sensitivity and specificity reported at a threshold chosen in advance.

---
## 10. Connections

- **Builds on**: nothing earlier in the curriculum; this is the first notebook. It uses school arithmetic and Pythagoras.
- **Leads to**:
  - 00.01.02 *Eigenvalues & Decomposition*: the principal directions of the biomarker cloud;
  - 00.01.03 *SVD & Low-Rank Approximation*: the singular values used in Experiment 3;
  - 00.06.01 *Stability, Precision & Solvers*: why the normal equations failed;
  - 03.03.01 *Standardisation, Normalisation & Robust Scaling*: Experiment 2 done properly, including leakage;
  - 04.02.02 *Multicollinearity & VIF*: the duplicated test, under an `explain` goal;
  - 05.02.01 *Distance Metrics & k Selection*: the neighbour vote as a classifier;
  - 05.08.01 *Centroid Classification*: the typical diseased profile as a rule.
- **Alternatives**:
  - Mahalanobis distance, which weights by the inverse covariance so that correlated tests are not double-counted;
  - learned metrics, which choose the weighting from data;
  - rank-based similarity for skewed results;
  - Gower distance for panels that mix numbers and categories.

---
## 11. Key Takeaways

1. A patient is a vector and a panel is a matrix: one matrix product makes every patient-to-patient comparison at once.
2. The dot product measures shared direction *and* size, cosine keeps only direction, and distance measures the gap. Choose by whether severity should count.
3. Units are weights. Standardise on the seen patients, or the column with the biggest numbers (here, age in years) decides who is "similar".
4. Check rank and condition number before trusting coefficients: a near-duplicate test sent them to ±90,000 while the predictions stayed put.
5. Similarity works only in a panel that is mostly signal: 300 uninformative tests pulled the neighbour vote from an AUC of 0.93 to 0.79.

---
## References

- Deisenroth, Faisal, Ong (2020). *Mathematics for Machine Learning*. `library:deisenroth2020`